In [1]:
import numpy as np
import pandas as pd
from Bio import Phylo
import sys
sys.path.append('../pysimARG')
from clonal_genealogy import ClonalTree
from newick_to_tree import newick_to_tree
from G4_test import G4_test
from LD import LD
from homoplasy_index import homoplasy_index
from Watterson_theta import Watterson_theta
from Tajima_pi import Tajima_pi
from Tajima_D import Tajima_D
from Wall_BQ import Wall_BQ
from Hudson_Rm import Hudson_Rm
from exp_regression import exp_regression
from segment_summary_stats import segment_summary_stats
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [2]:
genome_simbac = np.loadtxt("../data/SimBac/genomes_bool.csv", delimiter=",", dtype=bool)

In [3]:
genome_simbac.shape

(30, 1000000)

In [5]:
np.random.seed(100)
clonal_tree = ClonalTree(n=30)

# Load phylo tree and convert to ClonalTree format
phylo_tree = Phylo.read("../data/SimBac/clonal_frame.nwk", "newick")
Phylo.draw_ascii(phylo_tree)

edge, node_height = newick_to_tree(phylo_tree)
clonal_tree.edge = edge
clonal_tree.node_height = node_height
clonal_tree.height = np.max(node_height)
clonal_tree.length = np.sum(edge[:, 2])

                                                                    ____ 14
                                                                   |
                                                                   |    , 15
                                                        ___________|   _|
                                                       |           |  | , 5
                                                       |           |  | |
                                                       |           |__| | 24
                                                       |              |
                                                       |              | , 27
                                                       |              |_|
                                                       |                | 4
                                                       |
                                                       |      __________ 18
                                             

In [7]:
x_o_500 = np.full((10, 46), np.nan)
x_o_2000 = np.full((10, 46), np.nan)
x_o_6000 = np.full((10, 46), np.nan)

x_o_500, x_o_2000, x_o_6000

(array([[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan, nan, na

In [8]:
np.random.seed(500)
L = 500

for idx in range(10):
    start_pos = np.random.randint(1, genome_simbac.shape[1]-L+1)
    mat = genome_simbac[:, start_pos-1:start_pos+L-1]

    # Identify segregating sites
    has_true = mat.any(axis=0)
    has_false = ~mat.all(axis=0)
    idx_seg = np.where(has_true & has_false)[0]

    # Summary statistics LD r and G4 test
    seg_near, seg_far = 0, 0
    seg_20_50, seg_50_80 = 0, 0
    D_near, D_far, D_prime_near, D_prime_far, r2_near, r2_far = 0, 0, 0, 0, 0, 0
    g4_near, g4_far = 0, 0
    D_20_50, D_50_80, D_prime_20_50, D_prime_50_80, r2_20_50, r2_50_80 = 0, 0, 0, 0, 0, 0
    g4_20_50, g4_50_80 = 0, 0
    r_squares = []
    distances = []
    if idx_seg.size >= 2:
        for i in range(idx_seg.size - 1):
            for j in range(i + 1, idx_seg.size):
                dist_ij = idx_seg[j] - idx_seg[i]
                idx_pair = [idx_seg[i], idx_seg[j]]

                LD_result = LD(mat[:, idx_pair])
                r_sq = LD_result['r_square']
                r_squares.append(r_sq)
                distances.append(dist_ij)

                if dist_ij < L/2:
                    D_near += LD_result['D']
                    D_prime_near += LD_result['D_prime']
                    r2_near += LD_result['r_square']
                    g4_near += G4_test(mat[:, idx_pair])
                    seg_near += 1
                else:
                    D_far += LD_result['D']
                    D_prime_far += LD_result['D_prime']
                    r2_far += LD_result['r_square']
                    g4_far += G4_test(mat[:, idx_pair])
                    seg_far += 1
                if 20 <= dist_ij < 50:
                    D_20_50 += LD_result['D']
                    D_prime_20_50 += LD_result['D_prime']
                    r2_20_50 += LD_result['r_square']
                    g4_20_50 += G4_test(mat[:, idx_pair])
                    seg_20_50 += 1
                if 50 <= dist_ij <= 80:
                    D_50_80 += LD_result['D']
                    D_prime_50_80 += LD_result['D_prime']
                    r2_50_80 += LD_result['r_square']
                    g4_50_80 += G4_test(mat[:, idx_pair])
                    seg_50_80 += 1
        
        x_o_500[idx, 0] = D_near
        x_o_500[idx, 1] = D_far
        x_o_500[idx, 2] = D_prime_near
        x_o_500[idx, 3] = D_prime_far
        x_o_500[idx, 4] = r2_near
        x_o_500[idx, 5] = r2_far

        x_o_500[idx, 6] = g4_near
        x_o_500[idx, 7] = g4_far

        x_o_500[idx, 8] = D_near / seg_near if seg_near > 0 else 0
        x_o_500[idx, 9] = D_far / seg_far if seg_far > 0 else 0
        x_o_500[idx, 10] = D_prime_near / seg_near if seg_near > 0 else 0
        x_o_500[idx, 11] = D_prime_far / seg_far if seg_far > 0 else 0
        x_o_500[idx, 12] = r2_near / seg_near if seg_near > 0 else 0
        x_o_500[idx, 13] = r2_far / seg_far if seg_far > 0 else 0

        x_o_500[idx, 14] = g4_near / seg_near if seg_near > 0 else 0
        x_o_500[idx, 15] = g4_far / seg_far if seg_far > 0 else 0

        x_o_500[idx, 16] = D_20_50
        x_o_500[idx, 17] = D_50_80
        x_o_500[idx, 18] = D_prime_20_50
        x_o_500[idx, 19] = D_prime_50_80
        x_o_500[idx, 20] = r2_20_50
        x_o_500[idx, 21] = r2_50_80

        x_o_500[idx, 22] = g4_20_50
        x_o_500[idx, 23] = g4_50_80

        x_o_500[idx, 24] = D_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_500[idx, 25] = D_50_80 / seg_50_80 if seg_50_80 > 0 else 0
        x_o_500[idx, 26] = D_prime_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_500[idx, 27] = D_prime_50_80 / seg_50_80 if seg_50_80 > 0 else 0
        x_o_500[idx, 28] = r2_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_500[idx, 29] = r2_50_80 / seg_50_80 if seg_50_80 > 0 else 0

        x_o_500[idx, 30] = g4_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_500[idx, 31] = g4_50_80 / seg_50_80 if seg_50_80 > 0 else 0
    else:
        x_o_500[idx, :32] = 0
        x_o_500[idx, 42] = 0 # Kelly's Zns estimator
    
    # Summary statistic homoplasy index
    x_o_500[idx, 32] = homoplasy_index(clonal_tree, mat)

    # Summary statistic clade homoplasy index
    # s_vec[33] = clade_homoplasy(tree, node_site)

    # Summary statistic proportion of segregating sites
    count_S = idx_seg.size
    x_o_500[idx, 33] = count_S / L

    # Watterson's theta estimator
    x_o_500[idx, 34] = Watterson_theta(mat, count_S)

    # Tajima's pi estimators
    tajima_dict = Tajima_pi(mat, Wakeley=True)
    x_o_500[idx, 35] = tajima_dict['pi']
    x_o_500[idx, 36] = tajima_dict['pi2']

    # Tajima's D statistic
    x_o_500[idx, 37] = Tajima_D(mat, x_o_500[idx, 35], x_o_500[idx, 34], count_S)

    # Wall's B and Q statistics
    wall_dict = Wall_BQ(mat[:, idx_seg])
    x_o_500[idx, 38] = wall_dict['B']
    x_o_500[idx, 39] = wall_dict['Q']

    # Hudson's Rm estimator
    x_o_500[idx, 40] = Hudson_Rm(mat)

    # Kelly's Zns estimator
    x_o_500[idx, 41] = np.mean(r_squares)

    # Regression coefficient of r^2 on distance
    if len(r_squares) >= 2:
        df = pd.DataFrame({'x': distances, 'y': r_squares})
        mean_df = df.groupby('x')['y'].mean().reset_index()
        try:
            coeff = exp_regression(np.array(mean_df['x']), np.array(mean_df['y']))
            x_o_500[idx, 42:45] = coeff
        except RuntimeError as e:
            x_o_500[idx, 42:45] = 0
    else:
        x_o_500[idx, 42:45] = 0

    # Add the length of sequence as a summary statistic
    x_o_500[idx, 45] = L

    print(f"Completed {idx + 1} out of 100 simulations for L=500")

x_o_500

Completed 1 out of 100 simulations for L=500
Completed 2 out of 100 simulations for L=500
Completed 3 out of 100 simulations for L=500
Completed 4 out of 100 simulations for L=500
Completed 5 out of 100 simulations for L=500
Completed 6 out of 100 simulations for L=500
Completed 7 out of 100 simulations for L=500
Completed 8 out of 100 simulations for L=500
Completed 9 out of 100 simulations for L=500
Completed 10 out of 100 simulations for L=500


array([[ 5.47900000e+01,  1.27922222e+01,  2.51493461e+03,
         8.12188799e+02,  3.85188312e+02,  8.78186221e+01,
         5.03000000e+02,  3.32000000e+02,  1.96591317e-02,
         1.22883979e-02,  9.02380555e-01,  7.80200576e-01,
         1.38208939e-01,  8.43598675e-02,  1.80480804e-01,
         3.18924111e-01,  9.95222222e+00,  6.20777778e+00,
         3.61984144e+02,  3.45763742e+02,  7.74053996e+01,
         5.31887049e+01,  3.10000000e+01,  4.30000000e+01,
         2.65392593e-02,  1.68232460e-02,  9.65291051e-01,
         9.37029111e-01,  2.06414399e-01,  1.44142832e-01,
         8.26666667e-02,  1.16531165e-01,  5.72815534e-01,
         1.76000000e-01,  2.22129455e+01,  1.95701149e+01,
         1.44911751e+02, -4.53012481e-01,  2.29885057e-01,
         3.29545455e-01,  1.40000000e+01,  1.23565030e-01,
         2.62856104e-01,  2.52990409e-02,  8.78279899e-02,
         5.00000000e+02],
       [ 9.20422222e+01,  1.73733333e+01,  3.82736383e+03,
         1.11467156e+03,  5.98

In [9]:
x_o_500_new = np.full((10, 46), np.nan)

np.random.seed(500)
L = 500

for idx in range(10):
    start_pos = np.random.randint(1, genome_simbac.shape[1]-L+1)
    mat = genome_simbac[:, start_pos-1:start_pos+L-1]

    s_vec = segment_summary_stats(clonal_tree, mat)
    x_o_500_new[idx, :] = s_vec

    print(f"Completed {idx + 1} out of 100 simulations for L=500")

x_o_500_new

Completed 1 out of 100 simulations for L=500
Completed 2 out of 100 simulations for L=500
Completed 3 out of 100 simulations for L=500
Completed 4 out of 100 simulations for L=500
Completed 5 out of 100 simulations for L=500
Completed 6 out of 100 simulations for L=500
Completed 7 out of 100 simulations for L=500
Completed 8 out of 100 simulations for L=500
Completed 9 out of 100 simulations for L=500
Completed 10 out of 100 simulations for L=500


array([[ 5.47900000e+01,  1.27922222e+01,  2.51493461e+03,
         8.12188799e+02,  3.85188312e+02,  8.78186221e+01,
         5.03000000e+02,  3.32000000e+02,  1.96591317e-02,
         1.22883979e-02,  9.02380555e-01,  7.80200576e-01,
         1.38208939e-01,  8.43598675e-02,  1.80480804e-01,
         3.18924111e-01,  9.95222222e+00,  6.20777778e+00,
         3.61984144e+02,  3.45763742e+02,  7.74053996e+01,
         5.31887049e+01,  3.10000000e+01,  4.30000000e+01,
         2.65392593e-02,  1.68232460e-02,  9.65291051e-01,
         9.37029111e-01,  2.06414399e-01,  1.44142832e-01,
         8.26666667e-02,  1.16531165e-01,  5.72815534e-01,
         1.76000000e-01,  2.22129455e+01,  1.95701149e+01,
         1.44911751e+02, -4.53012481e-01,  2.29885057e-01,
         3.29545455e-01,  1.40000000e+01,  1.23565030e-01,
         2.62856104e-01,  2.52990409e-02,  8.78279899e-02,
         5.00000000e+02],
       [ 9.20422222e+01,  1.73733333e+01,  3.82736383e+03,
         1.11467156e+03,  5.98

In [10]:
x_o_500 == x_o_500_new

array([[ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  T

In [7]:
np.random.seed(2000)
L = 2000

for idx in range(100):
    start_pos = np.random.randint(1, genome_simbac.shape[1]-L+1)
    mat = genome_simbac[:, start_pos-1:start_pos+L-1]

    # Identify segregating sites
    has_true = mat.any(axis=0)
    has_false = ~mat.all(axis=0)
    idx_seg = np.where(has_true & has_false)[0]

    # Summary statistics LD r and G4 test
    seg_near, seg_far = 0, 0
    seg_20_50, seg_50_80 = 0, 0
    D_near, D_far, D_prime_near, D_prime_far, r2_near, r2_far = 0, 0, 0, 0, 0, 0
    g4_near, g4_far = 0, 0
    D_20_50, D_50_80, D_prime_20_50, D_prime_50_80, r2_20_50, r2_50_80 = 0, 0, 0, 0, 0, 0
    g4_20_50, g4_50_80 = 0, 0
    r_squares = []
    distances = []
    if idx_seg.size >= 2:
        for i in range(idx_seg.size - 1):
            for j in range(i + 1, idx_seg.size):
                dist_ij = idx_seg[j] - idx_seg[i]
                idx_pair = [idx_seg[i], idx_seg[j]]

                LD_result = LD(mat[:, idx_pair])
                r_sq = LD_result['r_square']
                r_squares.append(r_sq)
                distances.append(dist_ij)

                if dist_ij < L/2:
                    D_near += LD_result['D']
                    D_prime_near += LD_result['D_prime']
                    r2_near += LD_result['r_square']
                    g4_near += G4_test(mat[:, idx_pair])
                    seg_near += 1
                else:
                    D_far += LD_result['D']
                    D_prime_far += LD_result['D_prime']
                    r2_far += LD_result['r_square']
                    g4_far += G4_test(mat[:, idx_pair])
                    seg_far += 1
                if 20 <= dist_ij < 50:
                    D_20_50 += LD_result['D']
                    D_prime_20_50 += LD_result['D_prime']
                    r2_20_50 += LD_result['r_square']
                    g4_20_50 += G4_test(mat[:, idx_pair])
                    seg_20_50 += 1
                if 50 <= dist_ij <= 80:
                    D_50_80 += LD_result['D']
                    D_prime_50_80 += LD_result['D_prime']
                    r2_50_80 += LD_result['r_square']
                    g4_50_80 += G4_test(mat[:, idx_pair])
                    seg_50_80 += 1
        
        x_o_2000[idx, 0] = D_near
        x_o_2000[idx, 1] = D_far
        x_o_2000[idx, 2] = D_prime_near
        x_o_2000[idx, 3] = D_prime_far
        x_o_2000[idx, 4] = r2_near
        x_o_2000[idx, 5] = r2_far

        x_o_2000[idx, 6] = g4_near
        x_o_2000[idx, 7] = g4_far

        x_o_2000[idx, 8] = D_near / seg_near if seg_near > 0 else 0
        x_o_2000[idx, 9] = D_far / seg_far if seg_far > 0 else 0
        x_o_2000[idx, 10] = D_prime_near / seg_near if seg_near > 0 else 0
        x_o_2000[idx, 11] = D_prime_far / seg_far if seg_far > 0 else 0
        x_o_2000[idx, 12] = r2_near / seg_near if seg_near > 0 else 0
        x_o_2000[idx, 13] = r2_far / seg_far if seg_far > 0 else 0

        x_o_2000[idx, 14] = g4_near / seg_near if seg_near > 0 else 0
        x_o_2000[idx, 15] = g4_far / seg_far if seg_far > 0 else 0

        x_o_2000[idx, 16] = D_20_50
        x_o_2000[idx, 17] = D_50_80
        x_o_2000[idx, 18] = D_prime_20_50
        x_o_2000[idx, 19] = D_prime_50_80
        x_o_2000[idx, 20] = r2_20_50
        x_o_2000[idx, 21] = r2_50_80

        x_o_2000[idx, 22] = g4_20_50
        x_o_2000[idx, 23] = g4_50_80

        x_o_2000[idx, 24] = D_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_2000[idx, 25] = D_50_80 / seg_50_80 if seg_50_80 > 0 else 0
        x_o_2000[idx, 26] = D_prime_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_2000[idx, 27] = D_prime_50_80 / seg_50_80 if seg_50_80 > 0 else 0
        x_o_2000[idx, 28] = r2_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_2000[idx, 29] = r2_50_80 / seg_50_80 if seg_50_80 > 0 else 0

        x_o_2000[idx, 30] = g4_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_2000[idx, 31] = g4_50_80 / seg_50_80 if seg_50_80 > 0 else 0
    else:
        x_o_2000[idx, :32] = 0
        x_o_2000[idx, 42] = 0 # Kelly's Zns estimator
    
    # Summary statistic homoplasy index
    x_o_2000[idx, 32] = homoplasy_index(clonal_tree, mat)

    # Summary statistic clade homoplasy index
    # s_vec[33] = clade_homoplasy(tree, node_site)

    # Summary statistic proportion of segregating sites
    count_S = idx_seg.size
    x_o_2000[idx, 33] = count_S / L

    # Watterson's theta estimator
    x_o_2000[idx, 34] = Watterson_theta(mat, count_S)

    # Tajima's pi estimators
    tajima_dict = Tajima_pi(mat, Wakeley=True)
    x_o_2000[idx, 35] = tajima_dict['pi']
    x_o_2000[idx, 36] = tajima_dict['pi2']

    # Tajima's D statistic
    x_o_2000[idx, 37] = Tajima_D(mat, x_o_2000[idx, 35], x_o_2000[idx, 34], count_S)

    # Wall's B and Q statistics
    wall_dict = Wall_BQ(mat[:, idx_seg])
    x_o_2000[idx, 38] = wall_dict['B']
    x_o_2000[idx, 39] = wall_dict['Q']

    # Hudson's Rm estimator
    x_o_2000[idx, 40] = Hudson_Rm(mat)

    # Kelly's Zns estimator
    x_o_2000[idx, 41] = np.mean(r_squares)

    # Regression coefficient of r^2 on distance
    if len(r_squares) >= 2:
        df = pd.DataFrame({'x': distances, 'y': r_squares})
        mean_df = df.groupby('x')['y'].mean().reset_index()
        try:
            coeff = exp_regression(np.array(mean_df['x']), np.array(mean_df['y']))
            x_o_2000[idx, 42:45] = coeff
        except RuntimeError as e:
            x_o_2000[idx, 42:45] = 0
    else:
        x_o_2000[idx, 42:45] = 0

    # Add the length of sequence as a summary statistic
    x_o_2000[idx, 45] = L

    if (idx + 1) % 10 == 0:
        print(f"Completed {idx + 1} out of 100 simulations for L=2000")

x_o_2000

Completed 10 out of 100 simulations for L=2000


KeyboardInterrupt: 

In [ ]:
np.random.seed(6000)
L = 6000

for idx in range(100):
    start_pos = np.random.randint(1, genome_simbac.shape[1]-L+1)
    mat = genome_simbac[:, start_pos-1:start_pos+L-1]

    # Identify segregating sites
    has_true = mat.any(axis=0)
    has_false = ~mat.all(axis=0)
    idx_seg = np.where(has_true & has_false)[0]

    # Summary statistics LD r and G4 test
    seg_near, seg_far = 0, 0
    seg_20_50, seg_50_80 = 0, 0
    D_near, D_far, D_prime_near, D_prime_far, r2_near, r2_far = 0, 0, 0, 0, 0, 0
    g4_near, g4_far = 0, 0
    D_20_50, D_50_80, D_prime_20_50, D_prime_50_80, r2_20_50, r2_50_80 = 0, 0, 0, 0, 0, 0
    g4_20_50, g4_50_80 = 0, 0
    r_squares = []
    distances = []
    if idx_seg.size >= 2:
        for i in range(idx_seg.size - 1):
            for j in range(i + 1, idx_seg.size):
                dist_ij = idx_seg[j] - idx_seg[i]
                idx_pair = [idx_seg[i], idx_seg[j]]

                LD_result = LD(mat[:, idx_pair])
                r_sq = LD_result['r_square']
                r_squares.append(r_sq)
                distances.append(dist_ij)

                if dist_ij < L/2:
                    D_near += LD_result['D']
                    D_prime_near += LD_result['D_prime']
                    r2_near += LD_result['r_square']
                    g4_near += G4_test(mat[:, idx_pair])
                    seg_near += 1
                else:
                    D_far += LD_result['D']
                    D_prime_far += LD_result['D_prime']
                    r2_far += LD_result['r_square']
                    g4_far += G4_test(mat[:, idx_pair])
                    seg_far += 1
                if 20 <= dist_ij < 50:
                    D_20_50 += LD_result['D']
                    D_prime_20_50 += LD_result['D_prime']
                    r2_20_50 += LD_result['r_square']
                    g4_20_50 += G4_test(mat[:, idx_pair])
                    seg_20_50 += 1
                if 50 <= dist_ij <= 80:
                    D_50_80 += LD_result['D']
                    D_prime_50_80 += LD_result['D_prime']
                    r2_50_80 += LD_result['r_square']
                    g4_50_80 += G4_test(mat[:, idx_pair])
                    seg_50_80 += 1
        
        x_o_6000[idx, 0] = D_near
        x_o_6000[idx, 1] = D_far
        x_o_6000[idx, 2] = D_prime_near
        x_o_6000[idx, 3] = D_prime_far
        x_o_6000[idx, 4] = r2_near
        x_o_6000[idx, 5] = r2_far

        x_o_6000[idx, 6] = g4_near
        x_o_6000[idx, 7] = g4_far

        x_o_6000[idx, 8] = D_near / seg_near if seg_near > 0 else 0
        x_o_6000[idx, 9] = D_far / seg_far if seg_far > 0 else 0
        x_o_6000[idx, 10] = D_prime_near / seg_near if seg_near > 0 else 0
        x_o_6000[idx, 11] = D_prime_far / seg_far if seg_far > 0 else 0
        x_o_6000[idx, 12] = r2_near / seg_near if seg_near > 0 else 0
        x_o_6000[idx, 13] = r2_far / seg_far if seg_far > 0 else 0

        x_o_6000[idx, 14] = g4_near / seg_near if seg_near > 0 else 0
        x_o_6000[idx, 15] = g4_far / seg_far if seg_far > 0 else 0

        x_o_6000[idx, 16] = D_20_50
        x_o_6000[idx, 17] = D_50_80
        x_o_6000[idx, 18] = D_prime_20_50
        x_o_6000[idx, 19] = D_prime_50_80
        x_o_6000[idx, 20] = r2_20_50
        x_o_6000[idx, 21] = r2_50_80

        x_o_6000[idx, 22] = g4_20_50
        x_o_6000[idx, 23] = g4_50_80

        x_o_6000[idx, 24] = D_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_6000[idx, 25] = D_50_80 / seg_50_80 if seg_50_80 > 0 else 0
        x_o_6000[idx, 26] = D_prime_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_6000[idx, 27] = D_prime_50_80 / seg_50_80 if seg_50_80 > 0 else 0
        x_o_6000[idx, 28] = r2_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_6000[idx, 29] = r2_50_80 / seg_50_80 if seg_50_80 > 0 else 0

        x_o_6000[idx, 30] = g4_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_6000[idx, 31] = g4_50_80 / seg_50_80 if seg_50_80 > 0 else 0
    else:
        x_o_6000[idx, :32] = 0
        x_o_6000[idx, 42] = 0 # Kelly's Zns estimator
    
    # Summary statistic homoplasy index
    x_o_6000[idx, 32] = homoplasy_index(clonal_tree, mat)

    # Summary statistic clade homoplasy index
    # s_vec[33] = clade_homoplasy(tree, node_site)

    # Summary statistic proportion of segregating sites
    count_S = idx_seg.size
    x_o_6000[idx, 33] = count_S / L

    # Watterson's theta estimator
    x_o_6000[idx, 34] = Watterson_theta(mat, count_S)

    # Tajima's pi estimators
    tajima_dict = Tajima_pi(mat, Wakeley=True)
    x_o_6000[idx, 35] = tajima_dict['pi']
    x_o_6000[idx, 36] = tajima_dict['pi2']

    # Tajima's D statistic
    x_o_6000[idx, 37] = Tajima_D(mat, x_o_6000[idx, 35], x_o_6000[idx, 34], count_S)

    # Wall's B and Q statistics
    wall_dict = Wall_BQ(mat[:, idx_seg])
    x_o_6000[idx, 38] = wall_dict['B']
    x_o_6000[idx, 39] = wall_dict['Q']

    # Hudson's Rm estimator
    x_o_6000[idx, 40] = Hudson_Rm(mat)

    # Kelly's Zns estimator
    x_o_6000[idx, 41] = np.mean(r_squares)

    # Regression coefficient of r^2 on distance
    if len(r_squares) >= 2:
        df = pd.DataFrame({'x': distances, 'y': r_squares})
        mean_df = df.groupby('x')['y'].mean().reset_index()
        try:
            coeff = exp_regression(np.array(mean_df['x']), np.array(mean_df['y']))
            x_o_6000[idx, 42:45] = coeff
        except RuntimeError as e:
            x_o_6000[idx, 42:45] = 0
    else:
        x_o_6000[idx, 42:45] = 0

    # Add the length of sequence as a summary statistic
    x_o_6000[idx, 45] = L

    if (idx + 1) % 10 == 0:
        print(f"Completed {idx + 1} out of 100 simulations for L=6000")

x_o_6000

Completed 10 out of 100 simulations for L=6000
Completed 20 out of 100 simulations for L=6000
Completed 30 out of 100 simulations for L=6000
Completed 40 out of 100 simulations for L=6000
Completed 50 out of 100 simulations for L=6000
Completed 60 out of 100 simulations for L=6000
Completed 70 out of 100 simulations for L=6000
Completed 80 out of 100 simulations for L=6000
Completed 90 out of 100 simulations for L=6000
Completed 100 out of 100 simulations for L=6000


array([[2.84312013e+04, 9.45858658e+03, 2.89170000e+04, ...,
        3.81861575e-01, 1.29500000e-01, 6.00000000e+03],
       [2.64561315e+04, 7.58501511e+03, 2.10210000e+04, ...,
        3.68013758e-01, 1.22500000e-01, 6.00000000e+03],
       [3.04437076e+04, 8.59125506e+03, 3.32450000e+04, ...,
        3.93135725e-01, 1.29666667e-01, 6.00000000e+03],
       ...,
       [4.05275498e+04, 1.28291812e+04, 5.79210000e+04, ...,
        4.21943574e-01, 1.53666667e-01, 6.00000000e+03],
       [2.80704292e+04, 9.12998598e+03, 3.02520000e+04, ...,
        3.94223263e-01, 1.29333333e-01, 6.00000000e+03],
       [2.28071326e+04, 7.58467403e+03, 2.47000000e+04, ...,
        3.93327480e-01, 1.15166667e-01, 6.00000000e+03]], shape=(100, 19))

In [ ]:
np.savetxt('../data/x_500_mat.csv', x_o_500, delimiter=",")
np.savetxt('../data/x_2000_mat.csv', x_o_2000, delimiter=",")
np.savetxt('../data/x_6000_mat.csv', x_o_6000, delimiter=",")